<a href="https://colab.research.google.com/github/yusuke0000000/GCI_Competition2/blob/main/Competition2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive のマウント（Colab 上で自分のデータにアクセスするための設定）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 必要なライブラリのインストール
!pip install japanize-matplotlib lightgbm xgboost catboost optuna --quiet

In [ ]:
# モジュールのインポート
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib  # 日本語表示用
from sklearn.base import clone
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.exceptions import ConvergenceWarning
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
warnings.filterwarnings('ignore', category=ConvergenceWarning)

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/GCI/competition_2/data/Copy of train.csv')
test = pd.read_csv('/content/drive/MyDrive/GCI/competition_2/data/Copy of test.csv')
print('Train:', train.shape, ' Test:', test.shape)
train

In [ ]:
target_counts = train['target'].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(target_counts.index.astype(str), target_counts.values, color=['steelblue', 'coral'])
plt.xlabel('target')
plt.ylabel('人数')
plt.title('目的変数（target）の分布')
plt.show()

In [ ]:
target_pct = train['target'].value_counts(normalize=True) * 100
print(f"target=0（非反応）: {target_pct[0]:.1f}%")
print(f"target=1（反応）  : {target_pct[1]:.1f}%")

In [ ]:
numeric_cols = train.select_dtypes(include=['number']).columns
numeric_cols = [c for c in numeric_cols if c not in ['customer_id', 'target']]

fig, axes = plt.subplots(4, 5, figsize=(20, 14))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].hist(train[col].dropna(), bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
categorical_cols = ['education_level', 'marital_status']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, col in enumerate(categorical_cols):
    counts = train[col].value_counts()
    axes[i].bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, col in enumerate(categorical_cols):
    target_mean = train.groupby(col)['target'].mean().sort_values(ascending=False)
    axes[i].bar(target_mean.index, target_mean.values, color='coral', edgecolor='white')
    axes[i].set_title(f'{col}別のtarget平均')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_ylabel('target平均')
plt.tight_layout()
plt.show()

In [ ]:
# 数値列だけを取り出す（customer_id列は除く）
numeric_for_corr = train.select_dtypes(include=['number']).drop(columns=['customer_id'])

plt.figure(figsize=(14, 10))
sns.heatmap(numeric_for_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('相関行列ヒートマップ')
plt.tight_layout()
plt.show()

### 各カラムの意味

予測に使う前に、各列が何を表すのかを確認しておきましょう。

| カラム名 | 意味 |
|---|---|
| customer_id | 顧客ID（予測には使いません） |
| birth_year | 生まれ年 |
| education_level | 学歴 |
| marital_status | 配偶状況 |
| annual_income | 年収 |
| num_children | 子供の人数 |
| num_teenagers | ティーンエイジャーの人数 |
| registration_date | 顧客の登録日 |
| days_since_last_purchase | 最終購入からの経過日数 |
| spend_wines | ワインへの消費額 |
| spend_fruits | 果物への消費額 |
| spend_meat | 肉類への消費額 |
| spend_fish | 魚類への消費額 |
| spend_sweets | 菓子への消費額 |
| spend_gold | 金製品への消費額 |
| deals_purchases | 割引での購入回数 |
| web_purchases | Web サイト経由の購入回数 |
| catalog_purchases | カタログ経由の購入回数 |
| store_purchases | 実店舗での購入回数 |
| monthly_web_visits | 月あたりの Web サイト訪問回数 |
| has_complaint | 過去に苦情を申し立てたか（1=あり） |
| target | キャンペーンに反応したか（1=反応）。**これが予測対象です** |

`spend_` で始まる列は「その商品カテゴリにいくら使ったか」、`_purchases` で終わる列は「そのチャネルで何回買ったか」を表します。


In [ ]:
median_income = train['annual_income'].median()
for df in [train, test]:
    df['annual_income'] = df['annual_income'].fillna(median_income)

In [ ]:
for col in ['education_level', 'marital_status']:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

In [ ]:
for df in [train, test]:
    df.drop(columns=['customer_id'], inplace=True)
print('Train欠損:', train.isnull().sum().sum(), ' 列:', list(train.columns))

In [ ]:
# 特徴量(登録日数作成)
for df in [train, test]:
  df['registration_date'] = pd.to_datetime(df['registration_date'])
print(train['registration_date'].dtype)
print(train['registration_date'].min(), '~', train['registration_date'].max())

ref_date = train['registration_date'].max()
for df in[train, test]:
  df['tenure'] = (ref_date - df['registration_date']).dt.days

for df in [train, test]:
  df.drop(columns=['registration_date'], inplace=True)


In [ ]:
for df in [train, test]:
  df.drop(columns=['registration_data'], inplace=True)

In [ ]:
print(train['tenure'].describe())
train.info()

In [ ]:
X = train.drop(columns=['target'])
y = train['target']
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
train_auc = roc_auc_score(y_tr, rf.predict_proba(X_tr)[:, 1])
valid_auc = roc_auc_score(y_va, rf.predict_proba(X_va)[:, 1])
print(f'Train AUC = {train_auc:.4f}')
print(f'Valid AUC = {valid_auc:.4f}')

In [ ]:
for d in [None, 10, 5, 3]:
    m = RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42).fit(X_tr, y_tr)
    tr = roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1])
    va = roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])
    print(f'max_depth={str(d):>4}:  Train={tr:.4f}  Valid={va:.4f}  gap={tr-va:.4f}')

In [ ]:
def cv_auc(model, X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for tr_idx, va_idx in skf.split(X, y):
        m = clone(model)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        p = m.predict_proba(X.iloc[va_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[va_idx], p))
    return np.mean(scores), scores

rf = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=42)
baseline_cv, folds = cv_auc(rf, X, y)
print('各foldのAUC:', [f'{s:.4f}' for s in folds])
print(f'基準スコア（CV平均AUC）= {baseline_cv:.4f}')

In [ ]:
spend_cols = ['spend_wines', 'spend_fruits', 'spend_meat', 'spend_fish', 'spend_sweets', 'spend_gold']
for df in [train, test]:
    df['age'] = 2024 - df['birth_year']  # 基準年は便宜上の固定値。大小関係が保てればよい
    df['total_spend'] = df[spend_cols].sum(axis=1)  # spend_* 6列の合計

In [ ]:
for df in [train, test]:
    ch_total = df['web_purchases'] + df['catalog_purchases'] + df['store_purchases']
    df['spend_per_purchase'] = df['total_spend'] / ch_total
    df['spend_per_purchase'] = df['spend_per_purchase'].replace([np.inf, -np.inf], np.nan).fillna(0)

In [ ]:
for df in [train, test]:
    ch_total = df['web_purchases'] + df['catalog_purchases'] + df['store_purchases']
    df['deal_ratio'] = df['deals_purchases'] / ch_total
    # ↑ ここで ch_total が 0 の行は inf か NaN になっている
    df['deal_ratio'] = df['deal_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0)

In [ ]:
X = train.drop(columns=['target'])
y = train['target']
fe_cv, _ = cv_auc(rf, X, y)
print(f'特徴量追加後 CV AUC = {fe_cv:.4f}  （基準 {baseline_cv:.4f}, 差 {fe_cv - baseline_cv:+.4f}）')

In [ ]:
# 特徴量重要度: モデルがどの列をよく分岐に使ったかを見る
imp_model = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=42).fit(X, y)
importances = pd.Series(imp_model.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('特徴量重要度（RandomForest）')
plt.tight_layout()
plt.show()

In [ ]:
# 不均衡対策: 評価はAUC（順位指標）、ロジスティック回帰は class_weight='balanced' を使用
models = {
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=42),
    'LogisticRegression': make_pipeline(StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
    'LightGBM': LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
        random_state=42, verbose=-1),
    'MLP': make_pipeline(StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
        eval_metric='logloss', random_state=42),
    'CatBoost': CatBoostClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
        verbose=0),
}

cv_scores = {}
for name, model in models.items():
    mean_auc, _ = cv_auc(model, X, y)
    cv_scores[name] = mean_auc
    print(f'{name:20s} CV AUC = {mean_auc:.4f}')

In [ ]:
best_name = max(cv_scores, key=cv_scores.get)
print(f'\nこのデータで最も良かったモデル: {best_name}（CV AUC = {cv_scores[best_name]:.4f}）')

In [ ]:
# 改良
def cv_auc_ensemble(model_list, X, y, n_splits=5):
  skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
  scores = []
  for tr_idx, va_idx in skf.split(X, y):
    preds = np.zeros(len(va_idx))
    for model in model_list:
      m = clone(model).fit(X.iloc[tr_idx], y.iloc[tr_idx])
      preds += m.predict_proba(X.iloc[va_idx])[:, 1]
    preds /= len(model_list)
    scores.append(roc_auc_score(y.iloc[va_idx], preds))
  return np.mean(scores)

# LightGBM / XGBoost / RandomForest を選んだ理由
# 単体で強く、かつ学習アルゴリズムが異なるので「間違い方が分散」しやすい組み合わせ
ens_models = [models['LightGBM'], models['XGBoost'], models['RandomForest']]
ens_cv = cv_auc_ensemble(ens_models, X, y)
print(f'アンサンブル(単純平均) CV AUC = {ens_cv:.4f}')

In [ ]:
from sklearn.ensemble import StackingClassifier

stack = StackingClassifier(
    estimators=[('lgbm', models['LightGBM']), ('xgb', models['XGBoost']), ('rf', models['RandomForest'])],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
)
stack_cv, _ = cv_auc(stack, X, y)
print(f'スタッキング CV AUC = {stack_cv:.4f}')

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 600),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int('num_leaves', 7, 63),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        min_child_samples=trial.suggest_int('min_child_samples', 10, 80),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    )
    model = LGBMClassifier(**params, random_state=42, verbose=-1)
    mean_auc, _ = cv_auc(model, X, y)
    return mean_auc

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=True)  # 試行回数。増やすほど時間がかかる
print(f'Optuna ベスト CV AUC = {study.best_value:.4f}  （基準 {baseline_cv:.4f}, 差 {study.best_value - baseline_cv:+.4f}）')

In [ ]:
# ベスト設定で 5-fold それぞれ学習し、test 予測を平均する（k-fold アンサンブル）
X_test = test[X.columns]  # train と同じ特徴量・順序に揃える
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_pred = np.zeros(len(test))
for tr_idx, va_idx in skf.split(X, y):
    m = LGBMClassifier(**study.best_params, random_state=42, verbose=-1)
    m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    test_pred += m.predict_proba(X_test)[:, 1] / skf.n_splits
print('ベスト設定:', study.best_params)

In [ ]:
# ここまでの CV スコアを一覧で比較し、最も高いものを提出に使う
summary = {'RF基準(5章)': baseline_cv}
summary.update({f'{k}(7章)': v for k, v in cv_scores.items()})
summary['アンサンブル(8章)'] = ens_cv
summary['スタッキング(8章)'] = stack_cv
summary['Optunaチューニング(9章)'] = study.best_value
for k, v in sorted(summary.items(), key=lambda kv: -kv[1]):
    print(f'{k:24s} CV AUC = {v:.4f}')

In [ ]:
PATH = '/content/drive/MyDrive/GCI/competition_2/data/'
# 9章で作った test_pred（5-fold アンサンブルの平均予測）から提出ファイルを作る
submission = pd.read_csv(PATH + 'Copy of sample_submission.csv')
submission['target'] = test_pred
submission.to_csv(PATH + 'submission_advanced2.csv', index=False)
print('提出行数:', len(submission))
print('target範囲:', round(submission['target'].min(), 3), '〜', round(submission['target'].max(), 3))
submission.head()

In [ ]:
# Google Colab からダウンロードする場合
from google.colab import files
files.download(PATH + 'submission_advanced.csv')